# DublajUz — Chatterbox serveri (Colab)

1. **Runtime → Change runtime type → T4 GPU** ni tanlang.
2. Katakchalarni tartib bilan ishga tushiring.
3. Oxirgi katakcha **manzil** va **token** chiqaradi — ularni ilovada *Sozlamalar → Ovoz: Chatterbox va klonlash* ga kiriting.

Colab sessiyasi yopilsa, server ham to'xtaydi (qayta ishga tushirasiz, manzil o'zgaradi).

## 1) O'rnatish (5–10 daqiqa)

In [ ]:
import os
os.chdir('/content')
!git clone https://github.com/gokhaneraslan/chatterbox-finetuning.git
os.chdir('/content/chatterbox-finetuning')
!pip -q install -r requirements.txt
!pip -q install "setuptools<81" fastapi uvicorn python-multipart soundfile
!python setup.py


## 2) O'zbekcha model fayllari

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download('Abduqayum/uzbek-tts-natural-speech-chatterbox', local_dir='uzbek_chatterbox')
!cp uzbek_chatterbox/inference.py .


## 3) Server fayli (avtomatik yoziladi)

In [ ]:
%%writefile chatterbox_server.py
"""
DublajUz uchun Chatterbox (o'zbekcha fine-tune) TTS serveri.

Bu fayl `gokhaneraslan/chatterbox-finetuning` papkasining ildizida (inference.py yonida) ishga tushiriladi:
    CB_TOKEN=maxfiy_token uvicorn chatterbox_server:app --host 0.0.0.0 --port 8000

Endpointlar:
    GET  /health            -> {"ok": true, "device": "...", "sr": 24000}
    POST /tts               -> audio/wav (PCM16)
         form: text, exaggeration (ixtiyoriy), cfg_weight (ixtiyoriy), seed (ixtiyoriy)
         file: ref (ixtiyoriy) — klonlanadigan namuna ovoz; bo'lmasa standart ovoz
Barcha so'rovlarda `X-Api-Key: <CB_TOKEN>` sarlavhasi kerak.
"""
import hashlib
import io
import os
import secrets
import tempfile
import threading
from typing import Optional

import soundfile as sf
import torch
from fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile
from fastapi.responses import JSONResponse, Response

import inference as ci  # model repo'dagi inference.py (toolkit ildizida turishi kerak)

TOKEN = os.environ.get("CB_TOKEN") or secrets.token_urlsafe(16)
BASE = os.environ.get("CB_BASE", "pretrained_models")
WEIGHTS = os.environ.get("CB_WEIGHTS", "uzbek_chatterbox/t3_finetuned_merged.safetensors")
DEFAULT_REF = os.environ.get("CB_DEFAULT_REF", "uzbek_chatterbox/reference_voice.wav")
MAX_REF_BYTES = 25 * 1024 * 1024

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[server] model yuklanmoqda ({device}) ...", flush=True)
engine = ci.load_engine(BASE, WEIGHTS, device)
print("[server] tayyor. Token:", TOKEN, flush=True)

lock = threading.Lock()  # GPU'da bir vaqtda bitta sintez
ref_dir = tempfile.mkdtemp(prefix="cb_refs_")
app = FastAPI()


def _auth(key: Optional[str]):
    if not key or not secrets.compare_digest(key, TOKEN):
        raise HTTPException(status_code=401, detail="bad token")


@app.get("/health")
def health(x_api_key: Optional[str] = Header(default=None)):
    _auth(x_api_key)
    return JSONResponse({"ok": True, "device": device, "sr": int(engine.sr)})


@app.post("/tts")
def tts(
    text: str = Form(...),
    exaggeration: float = Form(0.5),
    cfg_weight: float = Form(0.5),
    seed: int = Form(42),
    ref: Optional[UploadFile] = File(None),
    x_api_key: Optional[str] = Header(default=None),
):
    _auth(x_api_key)
    text = (text or "").strip()
    if not text:
        raise HTTPException(status_code=400, detail="text bo'sh")

    ref_path = DEFAULT_REF
    if ref is not None:
        data = ref.file.read(MAX_REF_BYTES + 1)
        if len(data) > MAX_REF_BYTES:
            raise HTTPException(status_code=413, detail="ref juda katta")
        if data:
            digest = hashlib.sha1(data).hexdigest()
            suffix = os.path.splitext(ref.filename or "")[1][:6] or ".wav"
            ref_path = os.path.join(ref_dir, digest + suffix)
            if not os.path.exists(ref_path):
                with open(ref_path, "wb") as f:
                    f.write(data)

    with lock:
        try:
            audio, sr = ci.synthesize(
                engine, text, ref_path,
                exaggeration=exaggeration, cfg_weight=cfg_weight,
                repetition_penalty=2.0, temperature=0.8, seed=seed,
            )
        except Exception as e:  # noqa: BLE001
            raise HTTPException(status_code=500, detail=f"sintez xatosi: {e}")

    if audio is None or len(audio) < 100:
        raise HTTPException(status_code=500, detail="bo'sh audio")

    buf = io.BytesIO()
    sf.write(buf, audio, sr, format="WAV", subtype="PCM_16")
    return Response(content=buf.getvalue(), media_type="audio/wav")


## 4) Ishga tushirish va tunnel

In [ ]:
import os, re, secrets, subprocess, threading, time
os.environ['CB_TOKEN'] = secrets.token_urlsafe(16)

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

srv = subprocess.Popen(['python','-m','uvicorn','chatterbox_server:app','--host','0.0.0.0','--port','8000'],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
def pump():
    for line in srv.stdout:
        print('[srv]', line.rstrip())
threading.Thread(target=pump, daemon=True).start()

# model yuklanishini kutamiz
import urllib.request
for _ in range(180):
    try:
        urllib.request.urlopen('http://127.0.0.1:8000/docs', timeout=2); break
    except Exception:
        time.sleep(2)

tun = subprocess.Popen(['./cloudflared','tunnel','--url','http://127.0.0.1:8000'],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in tun.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0); break
print('\nSERVER MANZILI:', url)
print('TOKEN:', os.environ['CB_TOKEN'])
print('\nBu katakcha ishlab turishi kerak — to\'xtatmang.')
threading.Thread(target=lambda: [None for _ in tun.stdout], daemon=True).start()
